# Shared candidate pool

Issue: #21

## Question

The two rerankers (LightGBM, DCN-V2 with MMoE) must rank the **same** candidates. How do we build that pool, and how many relevant items does it contain?

## Data used

Inputs: the co-visitation ranker (#18), category popularity (#17), the ranking queries and shared rules (#10, `evalset.py`). Two-Tower candidates are added in #26.

Code: `notebooks/pool.py` (`build_pool`). Output: nothing saved; `build_pool` is used by #22 to #28.

**Why a shared pool.** If each reranker did its own retrieval, a difference in NDCG could come from retrieval (which items were considered) and not from ranking. With one pool, any difference is due to the ranker. The pool is also an upper limit: a relevant item that is not in the pool can never be found by a reranker.

## How the pool is built

- Retrievers now: **co-visitation** (top 200, only items with co-visitation evidence) and **category popularity** (top 200). A Two-Tower retriever is added later by passing one more list.
- Lists are merged by **ranks, not scores** (reciprocal rank fusion, `1 / (60 + rank)` summed over retrievers), because scores from different retrievers are not comparable.
- Prefix items and duplicates are removed. The pool is cut at **200** candidates.
- Each candidate keeps `source` (the retriever where it ranked best), `source_rank` (that rank), and its rank in every retriever.

In [ ]:
import resource
import sys
from pathlib import Path

import numpy as np
import polars as pl
from scipy import sparse

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from covis_rank import CovisRanker
from evalset import prepare_eval
from metrics import item_gains, ndcg_at_k, recall_at_k
from pool import build_pool
from split import WINDOWS, day_to_ms

DATA = Path("data") if Path("data").exists() else Path("../data")
TRAIN_END = day_to_ms(WINDOWS["w0 initial (weeks 1-2)"]["train_end_day"])
CAP = 200

## Toy checks

In [ ]:
prefix = {5}
r = {"covis": [1, 2, 5, 3, 1], "pop": [3, 4, 1, 6]}
pool = build_pool(prefix, r, cap=4)
aids = [c["aid"] for c in pool]
assert 5 not in aids                                  # prefix item removed
assert len(aids) == len(set(aids)) == 4               # no duplicates, cap respected
# item 1 is rank 1 in covis and rank 3 in pop (after removing 5, 3 is covis rank 3, so pop ranks: 3->1, 4->2, 1->3, 6->4)
by = {c["aid"]: c for c in pool}
assert by[1]["ranks"] == {"covis": 1, "pop": 3}
assert by[1]["source"] == "covis" and by[1]["source_rank"] == 1
assert abs(by[1]["rrf"] - (1 / 61 + 1 / 63)) < 1e-12
assert aids[0] == 1                                    # best fused score: covis 1 and pop 3
assert all({"source", "source_rank", "ranks", "rrf", "aid"} <= set(c) for c in pool)
assert build_pool(set(), {"a": [1, 1, 1]}, cap=10)[0]["ranks"] == {"a": 1}   # repeated items in a list count once
print("toy checks pass:", aids)

## Build the retrievers on the real data

In [ ]:
matrix = sparse.load_npz(DATA / "covis" / "covis.npz")
item_ids = np.load(DATA / "covis" / "item_ids.npy")
categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
item_cluster = dict(zip(categories["aid"].to_list(), categories["cluster"].to_list()))

events = pl.scan_parquet(DATA / "sample" / "events.parquet")
pop = events.filter((pl.col("type") == 0) & (pl.col("ts") < TRAIN_END)).group_by("aid").agg(pl.len().alias("clicks")).collect()
ranked = (categories.join(pop, on="aid", how="left").with_columns(pl.col("clicks").fill_null(0))
          .sort(["cluster", "clicks", "aid"], descending=[False, True, False]))
pop_by_cat = {c: part["aid"].to_numpy()[:2000] for (c,), part in ranked.group_by("cluster", maintain_order=True)}

covis = CovisRanker(matrix, item_ids, item_cluster, fill_by_cat=None, top=CAP)   # pure: no popularity fill


def covis_list(q):
    return covis.rank(q, pure=True)


def pop_list(q):
    out = []
    for a in pop_by_cat[q["category"]]:
        if int(a) not in q["prefix"]:
            out.append(int(a))
            if len(out) == CAP:
                break
    return out


queries = pl.read_parquet(DATA / "eval" / "ranking_queries.parquet")
prepared, stats = prepare_eval(queries)
print(stats)

## Build the pool for every query and measure recall

Recall@K of the pool is the share of relevant items among the first K candidates of the fused order (K = 20, 50, 100, 200). The two single retrievers are shown for comparison.

In [ ]:
KS = [20, 50, 100, 200]
rows = []
oracle = []
sizes = []
checks_ok = True
for q in prepared:
    cov, pp = covis_list(q), pop_list(q)
    pool = build_pool(q["prefix"], {"covis": cov, "pop": pp}, cap=CAP)
    aids = [c["aid"] for c in pool]
    # invariants
    assert len(aids) == len(set(aids)) <= CAP
    assert not (set(aids) & q["prefix"])
    assert all(c["source"] and c["source_rank"] >= 1 for c in pool)
    sizes.append(len(aids))
    relevant = set().union(*q["labels"].values())
    row = {}
    for k in KS:
        row[f"pool@{k}"] = recall_at_k(aids, relevant, k=k)
        row[f"covis@{k}"] = recall_at_k(cov, relevant, k=k)
        row[f"pop@{k}"] = recall_at_k(pp, relevant, k=k)
    rows.append(row)
    # best NDCG@10 a reranker could reach with this pool: relevant candidates first, best gain first
    gains = item_gains(q["labels"])
    in_pool = sorted((a for a in aids if a in gains), key=lambda a: -gains[a])
    oracle.append(ndcg_at_k(in_pool + [a for a in aids if a not in gains], q["labels"], k=10))
rec = pl.DataFrame(rows)
table = pl.DataFrame({
    "K": KS,
    "covis only": [rec[f"covis@{k}"].mean() for k in KS],
    "popularity only": [rec[f"pop@{k}"].mean() for k in KS],
    "pool (covis + popularity)": [rec[f"pool@{k}"].mean() for k in KS],
})
print(table)
print(f"pool size: min {min(sizes)}, median {int(np.median(sizes))}, max {max(sizes)} (cap {CAP})")
print(f"queries whose pool contains no relevant item: {(np.array(oracle) == 0).sum():,} of {len(oracle):,} ({(np.array(oracle) == 0).mean():.1%})")
print(f"best possible NDCG@10 with this pool (relevant candidates ranked first): {np.mean(oracle):.4f}")

In [ ]:
# sources of the candidates in the pool
src = {"covis": 0, "pop": 0}
both = 0
total = 0
for q in prepared[:5000]:
    for c in build_pool(q["prefix"], {"covis": covis_list(q), "pop": pop_list(q)}, cap=CAP):
        total += 1
        src[c["source"]] += 1
        both += len(c["ranks"]) == 2
print(f"first 5,000 queries: {total:,} candidates; best rank from covis {src['covis'] / total:.1%}, from popularity {src['pop'] / total:.1%}; found by both {both / total:.1%}")
assert rec["pool@200"].mean() >= max(rec["covis@200"].mean(), rec["pop@200"].mean()) - 1e-12   # a union never loses recall
print("peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Result

The table gives recall@20, 50, 100 and 200 for the pool and for each retriever.

- **Membership is what matters.** A reranker reorders the whole pool, so the number that limits it is the pool's recall@200 (about 0.61) and the best possible NDCG@10 of about 0.62. A third of the queries (33%) have no relevant item in the pool at all.
- **The fused order is not a ranking.** At small K the pool's recall is *lower* than covis alone (recall@20: 0.339 vs 0.362), because reciprocal rank fusion mixes popularity items into the top ranks. From K = 50 up the pool is better, and at 200 it is clearly better (0.608 vs 0.494 for covis alone, since covis has few in-category neighbours for many sessions). Only recall@200 is guaranteed to be at least as high as either retriever, and the notebook asserts that.
- Popularity supplies most of the candidates (80% have their best rank there), because covis returns short lists.

The Two-Tower retriever will be added in #26 and the table repeated.

## What I learned

To be written by the owner of the project.